In [3]:
import requests

In [16]:
from concurrent.futures import ThreadPoolExecutor

def send_request():
    r = requests.post(url)
    print(r.status_code)
    return r.status_code
    
    
with ThreadPoolExecutor(max_workers=50) as executor:
    futures = [executor.submit(send_request) for r in range(0, 10000)]


In [17]:
def get_problem(url):
    r = requests.get(url)
    return r.text

In [22]:
import requests
from datetime import datetime, timezone

def send_solution(id, code, cuname):
    url = ('https://codingbat.com/run')
    
    headers = {
        "Accept": "*/*",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "Accept-Language": "en-US,en;q=0.9",

        "Content-Type": "application/x-www-form-urlencoded",
        "Origin": "https://codingbat.com",
        "Referer": "https://codingbat.com/prob/p181646",
        "Priority": "u=0",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
        "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:151.0) Gecko/20100101 Firefox/151.0",
    }

    cookies = {
        "JSESSIONID": "7B4029F34F3A08CF0405FC2F1800B9C1"
    }

    data = {
        "id": id,
        "code": code,
        "cuname": cuname,
        "date":int(datetime.now(timezone.utc).timestamp()),
        "adate": datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%Sz").lower()
    }

    r = requests.post(
        url,
        headers=headers,
        cookies=cookies,
        data=data,
    )

    return r

text = send_solution("p154485", "def hello():\n    return 'Hello, World!'", "jayverma123@gmail.com")

In [20]:
from bs4 import BeautifulSoup
from pathlib import Path


def extract_next_link(html):
    soup = BeautifulSoup(html, 'html.parser')

    next_button = soup.find('a', string='next')
    if next_button:
        next_id = next_button.get('href')
        if next_id.startswith("/prob/"):
            next_id = next_id[len("/prob/"):]
            
        return next_id
    else:
        return None


def get_codingbat_problem(html):
    soup  = BeautifulSoup(html, 'html.parser')
    problem = soup.find('p', class_='max2')
    if problem:
        problem_text = problem.get_text(strip=True)
        return problem_text
    else:
        raise ValueError("No problem found")


def extract_coding_bat_problem_title(html):
    soup = BeautifulSoup(html, 'html.parser')
    title_element = soup.find_all('span', class_='h2')[1]
    if title_element:
        return title_element.get_text(strip=True)
    else:
        raise ValueError("No title found")
    

def get_solution_by_title(problem_title, answers_path="/home/jayv/Documents/CodingBat.Com-Agent/answers"):
    root = Path(answers_path)


    for file_path in root.rglob("*"):
        if file_path.is_file() and file_path.stem == problem_title:
            return file_path.read_text(encoding="utf-8")

    return None

In [23]:
base_url = "https://codingbat.com/prob/"
next_id = "p187868"


while next_id:
    page_content = get_problem(base_url + next_id)
    solution = get_solution_by_title(extract_coding_bat_problem_title(page_content))
    problem = get_codingbat_problem(page_content)
    status = send_solution(id=next_id, code=solution, cuname="jayverma123@gmail.com")
    next_id = extract_next_link(page_content)

In [3]:
from pathlib import Path
import shutil


source_root = Path("/home/jayv/Documents/CodingBat.Com-Agent/coding_bat_answers_repo")
dump_folder = Path("/home/jayv/Documents/CodingBat.Com-Agent/answers")

for file_path in source_root.rglob("*"):
    if file_path.is_file():
        dest = dump_folder / file_path.name
        
        shutil.move(str(file_path), str(dest))